<a href="https://colab.research.google.com/github/NaveenHerath/llm-from-scratch/blob/main/day9_tiny_gpt_feed_forward_and_residual_connections.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Real Transformer Block

Feed forward layer allows us to do the computation/thinking using the information gathered in the attention layer.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

batch_size = 32       # B: chunks per batch
block_size = 8        # T: the longest context the model can see
n_embd = 32           # C: numbers describing each character
n_head = 4            # heads per block
n_layer = 3           # new: how many blocks are stacked
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3

In [2]:
!wget -q -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [3]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [4]:
class Head(nn.Module):
    """ one head of masked self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                       # (B, T, hs)
        q = self.query(x)                                     # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5     # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)                                     # (B, T, hs)
        return wei @ v                                        # (B, T, hs)

In [5]:
class MultiHeadAttention(nn.Module):
    """ several heads of self-attention side by side """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, num_heads * head_size)
        out = self.proj(out)                                  # (B, T, n_embd)
        return out

In [6]:
class FeedForward(nn.Module):
    """ a small network applied to each position on its own """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),    # 32 -> 128
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),    # 128 -> 32
        )

    def forward(self, x):
        return self.net(x)

In [7]:
class Block(nn.Module):
    """ one transformer block: communication, then computation """

    def __init__(self, n_embd, n_head, residual=True):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.residual = residual

    def forward(self, x):
        if self.residual:
            x = x + self.sa(x)      # Run B: add attention's output onto x
            x = x + self.ffwd(x)    # then add feed-forward's output onto x
        else:
            x = self.sa(x)          # Run A: replace x
            x = self.ffwd(x)
        return x

In [8]:
class TinyGPT(nn.Module):

    def __init__(self, residual=True):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, residual) for _ in range(n_layer)])
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                   # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T))    # (T, C)
        x = tok_emb + pos_emb                                       # (B, T, C)
        x = self.blocks(x)                                          # (B, T, C)
        logits = self.lm_head(x)                                    # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [9]:
def train(model):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history, step_losses = [], []
    for step in range(max_iters + 1):
        if step % eval_interval == 0:
            l = estimate_loss(model)
            history.append((step, l['train'], l['val']))
            print(f"step {step:4d}: train {l['train']:.4f}, val {l['val']:.4f}")
        if step == max_iters:
            break
        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        step_losses.append(loss.item())
    return history, step_losses

In [10]:
torch.manual_seed(1337)
model_a = TinyGPT(residual=False)
print(sum(p.numel() for p in model_a.parameters()), 'parameters')
history_a, steps_a = train(model_a)

41921 parameters
step    0: train 4.1413, val 4.1429
step  500: train 3.1032, val 3.0960
step 1000: train 2.7789, val 2.7540
step 1500: train 2.5716, val 2.5646
step 2000: train 2.4882, val 2.4935
step 2500: train 2.4176, val 2.4225
step 3000: train 2.3665, val 2.3872
step 3500: train 2.3264, val 2.3295
step 4000: train 2.2829, val 2.2985
step 4500: train 2.2809, val 2.3003
step 5000: train 2.2366, val 2.2548


In [11]:
torch.manual_seed(1337)
model_b = TinyGPT(residual=True)
print(sum(p.numel() for p in model_b.parameters()), 'parameters')
history_b, steps_b = train(model_b)

41921 parameters
step    0: train 4.6328, val 4.6313
step  500: train 2.3720, val 2.3676
step 1000: train 2.2578, val 2.2622
step 1500: train 2.1725, val 2.1962
step 2000: train 2.1324, val 2.1744
step 2500: train 2.0992, val 2.1467
step 3000: train 2.0636, val 2.1351
step 3500: train 2.0486, val 2.1096
step 4000: train 2.0192, val 2.1001
step 4500: train 1.9981, val 2.0971
step 5000: train 1.9885, val 2.0675


In [12]:
torch.manual_seed(42)
start = torch.zeros((1, 1), dtype=torch.long)
print(decode(model_b.generate(start, max_new_tokens=300)[0].tolist()))



KIDYOLY ULIET:
Not mony chard'd, on anness mane they for and our magre,
The crake
that Gad,
Hlander'd:
The is u
ceasied famen, me't, hinge lalencily the faker no nated, wave if the stantiuniress:' but me besdied him, may and then like wuth
They home a fight and weres
Thesert wouch, by the must-s bu
